# Mission 3 — 2-4: Base 다중 seed 및 Base–Large 앙상블

목표는 **고정 임계값 0.5에서 내부 검증 Macro F1 0.66**입니다. 성능 향상과 목표 달성을 보장하지 않습니다.
서로 다른 모델의 오류를 확률 평균으로 보완하는 실험입니다. 새 학습 없이 기존 예측을 사용하므로 CPU 런타임으로 실행할 수 있습니다.

사전 고정 조건:
- A (주 실험): W1 Base seed 42·123·2026의 확률을 각각 1/3로 평균.
- B (추가 탐색): A의 확률 0.5 + W1 Large seed 42의 확률 0.5.
- 모든 예측은 `probability >= 0.5`. 가중치·임계값 탐색은 하지 않습니다.
- 각 원본 모델은 마지막 4 epoch의 예측입니다. Large 2 epoch 예측으로 바꾸지 않습니다.

Colab 파일 패널로 `/content`에 다음 세 파일을 업로드하세요.
1. `m3_targeted_weight_results.tar.gz` — Base seed 42
2. `m3_2_1_results.tar` — Base seed 123·2026
3. `m3_2_3_large_seed42_results.tar` — Large seed 42

학습 ZIP·Drive 연결·GPU는 필요하지 않습니다. 전체 셀을 순서대로 실행하면 작은 결과 TAR를 다운로드합니다.
이미 여러 실험에 사용한 내부 검증셋의 탐색 결과입니다. 최종 성능 확인에는 별도의 평가가 필요합니다.
예측 배열만으로 새 데이터 추론은 할 수 없습니다. 채택 시 구성 모델 3개 또는 4개의 가중치와 동일 전처리가 필요합니다.


## 1. 파일과 고정 실험 설정 확인


In [ ]:
from pathlib import Path
import io, json, tarfile, hashlib, platform, tempfile
from datetime import datetime, timezone
import numpy as np

ROOT = Path('/content')
ARCHIVES = {
    'old': ROOT / 'm3_targeted_weight_results.tar.gz',
    'multi': ROOT / 'm3_2_1_results.tar',
    'large': ROOT / 'm3_2_3_large_seed42_results.tar',
}
for path in ARCHIVES.values():
    assert path.is_file(), f'파일 패널로 업로드하세요: {path}'
TARGETS = ['고열','구토','두통','복통','어지러움','열상','오심','전신쇠약','호흡곤란']
SOURCES = {
    'base42': ('old', 'w1_targeted_low_recall_x125', 42),
    'base123': ('multi', 'seed_123/w1_targeted_low_recall_x125', 123),
    'base2026': ('multi', 'seed_2026/w1_targeted_low_recall_x125', 2026),
    'large42': ('large', 'seed_42/w1_encoder', 42),
}
BLENDS = {
    'A_base3_equal': {'base42':1/3, 'base123':1/3, 'base2026':1/3},
    'B_base3_large_equal': {'base42':1/6, 'base123':1/6, 'base2026':1/6, 'large42':.5},
}
OUT = Path(tempfile.mkdtemp(prefix='m3_2_4_', dir=ROOT))
plan = {'threshold':.5, 'primary':'A_base3_equal', 'secondary':'B_base3_large_equal',
        'blends':BLENDS, 'selection':'all constituent models use final epoch',
        'target_macro_f1':.66, 'evaluation':'internal validation',
        'created_utc':datetime.now(timezone.utc).isoformat()}
(OUT / 'experiment_plan.json').write_text(json.dumps(plan, ensure_ascii=False, indent=2))
print('결과 저장 폴더:', OUT)
print('고정 결합 비율:', BLENDS)


## 2. 예측값·정답·분할 일치 검증


In [ ]:
def read_member(archive, name):
    member = archive.getmember(name)
    assert member.isfile() and member.size < 20_000_000, f'잘못된 항목: {name}'
    return archive.extractfile(member).read()

data, configs, provenance = {}, {}, {}
for name, (key, prefix, seed) in SOURCES.items():
    with tarfile.open(ARCHIVES[key], 'r:*') as archive:
        raw = read_member(archive, prefix + '/dev_predictions.npz')
        config = json.loads(read_member(archive, prefix + '/run_config.json'))
    with np.load(io.BytesIO(raw), allow_pickle=False) as z:
        item = {k:z[k].copy() for k in ['probs','y_true','file_names','targets']}
    assert item['probs'].shape == item['y_true'].shape == (5792, 9)
    assert np.isfinite(item['probs']).all()
    assert ((item['probs'] >= 0) & (item['probs'] <= 1)).all()
    assert np.isin(item['y_true'], [0,1]).all()
    assert list(item['targets']) == TARGETS
    assert len(np.unique(item['file_names'])) == 5792
    assert config.get('seed', config['training']['seed']) == seed and config['selection'] == 'final_epoch'
    assert config['threshold'] == .5 and config['loss'] == 'weighted_bce'
    assert config['model'] == ('klue/roberta-large' if name == 'large42' else 'klue/roberta-base')
    data[name], configs[name] = item, config
    provenance[name] = {'archive':ARCHIVES[key].name, 'member':prefix,
                        'prediction_sha256':hashlib.sha256(raw).hexdigest(),
                        'run_config':config}

reference = data['base42']
for name, item in data.items():
    for key in ['y_true','file_names','targets']:
        assert np.array_equal(item[key], reference[key]), f'{name}: {key}불일치'
    for key in ['manifest_sha256','input','preprocessing','head']:
        assert configs[name][key] == configs['base42'][key], (name, key)
    np.testing.assert_allclose(configs[name]['pos_weight'], configs['base42']['pos_weight'], rtol=0, atol=1e-7)
(OUT / 'source_provenance.json').write_text(json.dumps(provenance,ensure_ascii=False,indent=2))
print('4개 모델의 5,792개 정답·행 순서·가중치 일치 확인')


## 3. 고정 임계값 평가 함수 검사


In [ ]:
import csv
def scores(y, probs):
    y = np.asarray(y, dtype=bool)
    pred = np.asarray(probs) >= .5
    assert y.shape == pred.shape
    tp = (y & pred).sum(0); fp = (~y & pred).sum(0); fn = (y & ~pred).sum(0)
    def divide(a,b):
        return np.divide(a,b,out=np.zeros_like(a,dtype=float),where=b!=0)
    precision=divide(tp,tp+fp); recall=divide(tp,tp+fn); f1=divide(2*tp,2*tp+fp+fn)
    return {'macro_f1':float(f1.mean()),'precision':precision,'recall':recall,'f1':f1,
            'tp':tp,'fp':fp,'fn':fn,'support':y.sum(0)}

test = scores(np.array([[1,0],[0,1]]),np.array([[.5,.1],[.1,.9]]))
assert test['macro_f1'] == 1.0
assert scores(np.zeros((2,2)),np.zeros((2,2)))['macro_f1'] == 0.0

def write_csv(path, records):
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(records[0]))
        writer.writeheader(); writer.writerows(records)
print('임계값 0.5·F1 계산 검사 통과')


## 4. A·B 앙상블과 단일 모델 비교

학습 seed별 평균 점수와 확률을 평균한 앙상블 점수는 서로 다릅니다. 아래는 후자를 실제 계산합니다.


In [ ]:
predictions = {name:item['probs'].astype(np.float64) for name,item in data.items()}
for name, weights in BLENDS.items():
    assert abs(sum(weights.values())-1)<1e-12 and all(w>=0 for w in weights.values())
    predictions[name] = sum(weight*predictions[key] for key,weight in weights.items())

y = reference['y_true']
evaluated = {name:scores(y,probs) for name,probs in predictions.items()}
summary, per_label = [], []
for name, result in evaluated.items():
    summary.append({'experiment':name,'macro_f1':result['macro_f1'],
                    'delta_vs_base42':result['macro_f1']-evaluated['base42']['macro_f1'],
                    'threshold':.5,'target_reached':result['macro_f1']>=.66})
    for i,symptom in enumerate(TARGETS):
        per_label.append({'experiment':name,'symptom':symptom,
                          **{k:float(result[k][i]) for k in ['precision','recall','f1']},
                          **{k:int(result[k][i]) for k in ['tp','fp','fn','support']}})
    if name in BLENDS:
        np.savez_compressed(OUT / f'{name}_predictions.npz',probs=predictions[name],y_true=y,
                            file_names=reference['file_names'],targets=reference['targets'])
write_csv(OUT/'summary.csv',summary)
write_csv(OUT/'per_label.csv',per_label)
print(f"{'실험':25s} {'Macro F1':>10s} {'Base42 대비':>12s}")
for r in summary:
    print(f"{r['experiment']:25s} {r['macro_f1']:10.6f} {r['delta_vs_base42']:+12.6f}")
print('\n주 실험 A의 목표 0.66 달성:', evaluated['A_base3_equal']['macro_f1'] >= .66)
print('추가 탐색 B의 목표 0.66 달성:', evaluated['B_base3_large_equal']['macro_f1'] >= .66)
print('두 결과를 모두 보고합니다. 내부 검증 탐색 결과이며 공식 평가가 아닙니다.')


## 5. 결과 보고서와 TAR 다운로드


In [ ]:
lines=['# 2-4 고정 비율 앙상블 결과','',
       '내부 검증 5,792건 / threshold 0.5 / 사전 고정 결합 비율.', '',
       '|실험|Macro F1|Base42 대비|','|---|---:|---:|']
for r in summary:
    lines.append(f"|{r['experiment']}|{r['macro_f1']:.6f}|{r['delta_vs_base42']:+.6f}|")
lines += ['', 'A는 Base 3개 seed 동일 비율 평균, B는 A와 Large를 50:50으로 평균했다.',
          '모든 구성 모델은 최종 4 epoch 모델이다. TF-IDF는 이 실험에 포함하지 않았다.',
          '같은 내부 검증셋을 반복 사용한 탐색 결과다. 최종 모델 선택 후 독립 평가가 필요하다.',
          '예측 배열에는 개별 통화 파일명이 있다. 공개 저장소에는 집계 지표와 설정을 공유한다.',
          '새 데이터 추론에는 구성 모델들의 가중치가 필요하다. 이 TAR에는 가중치가 없다.']
(OUT/'REPORT.md').write_text('\n'.join(lines),encoding='utf-8')
(OUT/'environment.json').write_text(json.dumps({'python':platform.python_version(),'numpy':np.__version__},indent=2))
archive_path=ROOT/'m3_2_4_fixed_ensemble_results.tar'
with tarfile.open(archive_path,'w') as archive:
    for path in sorted(OUT.iterdir()):
        if path.is_file(): archive.add(path,arcname=path.name)
print('완료:',archive_path,'MB:',round(archive_path.stat().st_size/1e6,2))
from google.colab import files
files.download(str(archive_path))
